# 07 · Ajuste Conjuntural com o Novo CAGED

**Objetivo:** complementar o risco *estrutural* da RAIS com a *conjuntura recente*
dos fluxos mensais do Novo CAGED, via um **fator de ajuste** por célula
L = (CBO 2díg × CNAE 2díg × UF).

**Entradas:** `data/processed/rates/level_completo.parquet` (estoque RAIS) e
microdados Novo CAGED (`config.caged`).  
**Saídas:** `data/processed/caged_fator_<motivo>.parquet`; figura/tabela em `outputs/`.

**Método:** a RAIS não tem estoque infra-anual; o CAGED não tem estoque nem tempo
de vínculo. Combinamos: `hazard_recente_L = (desligamentos_CAGED / meses)·12 / estoque_RAIS_L`,
e `fator_L = hazard_recente_L / hazard_estrutural_L` (suavizado p/ 1 em células ralas).
No scoring, `risco_ajustado = risco_estrutural × fator_L`.

**Premissas/Limitações:** o ajuste é *agregado* em L (não usa tempo de vínculo,
que o CAGED não traz); o mapa de `tipomovimentação` aproxima os motivos; UF e
CNAE/CBO devem ser comparáveis entre RAIS e CAGED (CNAE 2.0 subclasse, CBO 2002).

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
from src import io_utils, caged, rates, scoring
raw = cfg['abs']['raw']; motivos = cfg['motivos']; ufs = cfg.get('ufs_subset')
ano = cfg['caged']['ano']; meses = cfg['caged']['meses']
base = cfg['urls']['caged_base']

# Baixa, extrai, conta desligamentos e descarta o .txt — mês a mês.
acc = None; meses_proc = 0
for mm in meses:
    comp = f'{ano}{mm:02d}'
    z = raw / 'CAGED' / str(ano) / f'CAGEDMOV{comp}.7z'
    url = f'{base}/{ano}/{comp}/CAGEDMOV{comp}.7z'
    try:
        io_utils.download_ftp(url, z)
    except Exception as e:
        print(f'  {comp}: indisponível ({e}); pulando'); continue
    extr = io_utils.extract_7z(z, raw / 'CAGED' / str(ano))
    txt = next(p for p in extr if p.suffix.lower() == '.txt')
    tab, _ = caged.count_caged_deslig([txt], motivos, ufs)
    txt.unlink()
    acc = tab if acc is None else (pd.concat([acc, tab])
          .groupby(['cbo2','cnae2','uf'], as_index=False).sum())
    meses_proc += 1
    print(f'  {comp}: ok (acumulado: {len(acc):,} células L)', flush=True)
print('meses processados:', meses_proc)

In [ ]:
# Estoque estrutural RAIS por L=(cbo2,cnae2,uf) a partir do nível mais
# granular disponível que contenha cnae2 (robusto à exclusão de níveis).
tables, meta = rates.load_level_tables(cfg['abs']['rates'])
fonte_L = 'cbo2_cnae2_tempo_uf' if 'cbo2_cnae2_tempo_uf' in tables else meta['order'][-1]
rais_L = caged.rais_estoque_por_L(tables[fonte_L], motivos)
print('células L na RAIS:', f'{len(rais_L):,}')

# Fator conjuntural para o motivo default (dispensa s/ justa causa)
MOT = cfg['motivo_default'][0]
fat = caged.fator_ajuste_conjuntural(acc, meses_proc, rais_L, MOT,
                                     m_suav=cfg['caged']['shrinkage_fator'],
                                     n_anos_rais=len(cfg['anos']))
fat[['cbo2','cnae2','uf','fator']].to_parquet(
    cfg['abs']['processed'] / f'caged_fator_{MOT}.parquet', index=False)
print('fator salvo. Resumo do fator (peso conjuntural):')
display(fat['fator'].describe().round(3))

In [ ]:
# Onde a conjuntura recente mais AGRAVOU o risco vs a estrutura (fator alto)
alto = fat[fat['n'] >= 500].sort_values('fator', ascending=False).head(12)
alto.to_csv(cfg['abs']['tables'] / 'caged_fatores_top.csv', index=False)
display(alto[['cbo2','cnae2','uf','hazard_estrut','hazard_recente','fator','n']].round(4))

In [ ]:
# Demonstração: score SEM vs COM ajuste conjuntural (recarrega fatores)
sc = scoring.Scorer(cfg=cfg)   # nova instância já carrega caged_fator_*
exemplo = dict(cbo='521110', cnae='4711301', uf='SP', idade=35,
               escolaridade='medio_completo', tempo_vinculo_meses=8, tamanho_empresa=3)
for ajuste in (False, True):
    r = sc.score_pessoa(**exemplo, motivos=[cfg['motivo_default'][0]], ajuste_conjuntural=ajuste)
    m = cfg['motivo_default'][0]
    print(f'ajuste={ajuste!s:5} -> risco12m={r["risco"][m][12]:.3f} '
          f'fator={r["fator_conjuntural"][m]:.3f} hazard={r["hazard_anual"][m]:.3f}')

> **Leitura:** `fator > 1` significa que o CAGED recente aponta **mais**
> desligamentos involuntários naquela célula L do que a média estrutural da RAIS
> (conjuntura piorando); `fator < 1`, o oposto. O ajuste é deliberadamente
> conservador (suavizado p/ 1 e limitado ao intervalo [0,2; 5,0]).